In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import torch.distributed as dist
from datetime import timedelta
import torch.nn.functional as F
import numpy as np
import os
import random

In [12]:
def init(rank, world_size, backend='gloo'):
    os.environ['GLOO_SOCKET_IFNAME'] = 'eth0'
    os.environ['MASTER_ADDR'] = 'c1'
    os.environ['MASTER_PORT'] = '29500'

    dist.init_process_group(
        backend=backend,
        world_size=world_size,
        rank=rank,
        timeout=timedelta(seconds=60),
    )

    print(f"Rank {rank}: is_initialized and ready to communicate...")
    return dist.is_initialized()

def recv(arr):
    dist.recv(tensor=arr, src=1)

def send(arr):
    dist.send(tensor=arr, dst=1)

In [13]:
class TrainDataset(Dataset):
    def __init__(self,transform=None,path='/app/X_train.npy'):
        self.x= np.load(path)
        self.n = self.x.shape[0]
        self.height = self.x.shape[1]
        self.width = self.x.shape[2]
        self.transform =transform
        
    def __getitem__(self, index):
        sample_temp = self.x[index]
        
        if self.transform:
            sample_temp = self.transform(sample_temp)
            
        sample = sample_temp,index
        return sample
    
    def __len__(self):
        return self.n
    
    def shape(self):
        return self.n,self.height,self.width
    

class ToTensor:
    def __call__(self,input):
        input = input.reshape((-1,28,28)).astype('float32')
        return torch.from_numpy(input)
            

In [14]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        
    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        return x

In [15]:
random.seed(43)
np.random.seed(43)
torch.manual_seed(43)

batch_size = 250
transform = ToTensor()
dataset = TrainDataset(transform=transform)
dataloader = DataLoader(dataset=dataset,shuffle=False,batch_size=batch_size)

model = CNN()
optim = torch.optim.Adam(params=model.parameters(),lr=0.005)

saved_model_state = torch.load('/app/model.pth')
model.conv1.load_state_dict({"weight":saved_model_state['conv1.weight'],
                             "bias":saved_model_state["conv1.bias"]})
model.eval()

CNN(
  (conv1): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
)

In [16]:
def run(num_epoch,batch_size=1,model=model,optim=optim,dataloader=dataloader,send=send,recv=recv):
    for epoch in range(num_epoch):
        tot_gradient = []
        for i, (data, index) in enumerate(dataloader):
            output = model(data)
            send(output)
            
            gradient = torch.zeros_like(output)
            recv(gradient)
            
            output.backward(gradient)
            optim.step()
            
            optim.zero_grad()
            print(f"epoch {epoch} {i}/{len(dataloader)} completed...")
            
            
                
                
            

In [17]:
num_epoch = 50
init(0,2)
run(num_epoch,batch_size=batch_size)

dist.destroy_process_group()

Rank 0: is_initialized and ready to communicate...
epoch 0 0/240 completed...
epoch 0 1/240 completed...
epoch 0 2/240 completed...
epoch 0 3/240 completed...
epoch 0 4/240 completed...
epoch 0 5/240 completed...
epoch 0 6/240 completed...
epoch 0 7/240 completed...
epoch 0 8/240 completed...
epoch 0 9/240 completed...
epoch 0 10/240 completed...
epoch 0 11/240 completed...
epoch 0 12/240 completed...
epoch 0 13/240 completed...
epoch 0 14/240 completed...
epoch 0 15/240 completed...
epoch 0 16/240 completed...
epoch 0 17/240 completed...
epoch 0 18/240 completed...
epoch 0 19/240 completed...
epoch 0 20/240 completed...
epoch 0 21/240 completed...
epoch 0 22/240 completed...
epoch 0 23/240 completed...
epoch 0 24/240 completed...
epoch 0 25/240 completed...
epoch 0 26/240 completed...
epoch 0 27/240 completed...
epoch 0 28/240 completed...
epoch 0 29/240 completed...
epoch 0 30/240 completed...
epoch 0 31/240 completed...
epoch 0 32/240 completed...
epoch 0 33/240 completed...
epoch 0

In [18]:
# for name, param in model.named_parameters():
#     print(f"{name}: {param}")